# 01 — Exploratory Data Analysis
Download raw data from Kaggle, inspect it, and visualise distributions, correlations and churn patterns.

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Download dataset from Kaggle
path = kagglehub.dataset_download("gauravtopre/bank-customer-churn-dataset")
data = pd.read_csv(path + "/Bank-Customer-Churn-Prediction.csv")

# Save raw copy
data.to_csv("../data/raw/Bank-Customer-Churn-Prediction.csv", index=False)
print(f"Shape: {data.shape}")
data.head(3)

In [ ]:
# Overview: dtypes, stats, missing values
print(data.info())
print()
print(data.describe())
print()
print("Missing values:\n", data.isnull().sum())

In [ ]:
# Distributions of all numerical features
numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    if i < len(axes):
        axes[i].hist(data[col], bins=30, color='skyblue', edgecolor='black')
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

for i in range(len(numerical_cols), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Distributions of categorical features
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

fig, axes = plt.subplots(1, len(categorical_cols), figsize=(16, 6))
if len(categorical_cols) == 1:
    axes = [axes]

for i, col in enumerate(categorical_cols):
    data[col].value_counts().plot(kind='bar', color='steelblue', edgecolor='black', ax=axes[i])
    axes[i].set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', alpha=0.3)
    print(f"{col} percentages:\n", data[col].value_counts(normalize=True).mul(100).round(2))

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features only)
corr = data.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by country and gender
if {'gender', 'country'}.issubset(data.columns):
    pivot = data.pivot_table(values='churn', index='country', columns='gender', aggfunc='mean') * 100

    plt.figure(figsize=(6, 4))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
                cbar_kws={'label': 'Churn Rate (%)'}, linewidths=1, linecolor='black')
    plt.title('Churn Rate by Country and Gender', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(pivot)

In [ ]:
# Churn class distribution
churn_counts = data['churn'].value_counts()
print("Counts:\n", churn_counts)
print("\nPercentages:\n", churn_counts.div(len(data)).mul(100).round(2))

plt.figure(figsize=(6, 5))
plt.pie(churn_counts, labels=['Not Churned (0)', 'Churned (1)'],
        colors=['lightgreen', 'lightcoral'], autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 12})
plt.title('Churn Distribution', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: numerical features vs churn status
num_no_churn = [c for c in data.select_dtypes(include=[np.number]).columns if c != 'churn']

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(num_no_churn):
    if i < len(axes):
        sns.boxplot(data=data, x='churn', y=col, ax=axes[i], palette=['green', 'red'])
        axes[i].set_title(f'{col} by Churn Status')
        axes[i].set_xlabel('Churn (0=No, 1=Yes)')

for i in range(len(num_no_churn), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Churn count per categorical feature
for col in categorical_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=data, x=col, hue='churn', palette=['green', 'red'])
    plt.title(f'Churn by {col}', fontsize=14, fontweight='bold')
    plt.legend(title='Churn', labels=['Not Churned (0)', 'Churned (1)'])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print(pd.crosstab(data[col], data['churn']))

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(df, col):
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lo) | (df[col] > hi)]
    return outliers, lo, hi

for col in num_no_churn:
    outliers, lo, hi = detect_outliers_iqr(data, col)
    pct = len(outliers) / len(data) * 100
    print(f"{col}: {len(outliers)} outliers ({pct:.2f}%)  |  bounds [{lo:.2f}, {hi:.2f}]")

In [ ]:
# Export full dataset for Power BI
data.to_csv("../results/data_for_powerbi.csv", index=False)
print(f"Exported {data.shape[0]} rows, {data.shape[1]} columns")